---
title: "Data Collection"
format:
    html: 
        code-fold: false
---


{{< include overview.qmd >}} 

{{< include methods.qmd >}} 

# Code 

## Alerts 

In the following code, we collected csv files from the MBTA open source data portal. From there, we explored the files manually to see if there are inconsistencies with features or formatting. Thankfully, formatting and information was consistent. Then, we merged the files and added a year column for easy reference in analysis and visualizations. Finally, we used pandas to store collected data as a dataframe.  
 
- Why decide to merge all files instead of incrementally cleaning them? I personally found this method to produce less code, which is helpful in large project like this. 

- What are the drawbacks of this method? Less familiarization with individual datasets. 


**Libraries**:

In [5]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np 
import os 
import missingno as msno
import json
from serpapi import GoogleSearch


**Before Merging**:
Below is one example of the file structure before merging.  

In [19]:
example_data_2024 = "../../data/raw-data/Alerts_2024/2024-02_ALERTS.csv"

df1 = pd.read_csv(example_data_2024)
shape = df1.shape
print(df1.head(5))
print(shape) 

   alert_id        cause cause_detail  effect effect_detail severity_level  \
0    483702  MAINTENANCE  MAINTENANCE  DETOUR        DETOUR         SEVERE   
1    483702  MAINTENANCE  MAINTENANCE  DETOUR        DETOUR         SEVERE   
2    483702  MAINTENANCE  MAINTENANCE  DETOUR        DETOUR         SEVERE   
3    483702  MAINTENANCE  MAINTENANCE  DETOUR        DETOUR         SEVERE   
4    483702  MAINTENANCE  MAINTENANCE  DETOUR        DETOUR         SEVERE   

   severity                                             header description  \
0         7  Route 117 inbound trips all day on Saturdays a...         NaN   
1         7  Route 117 inbound trips all day on Saturdays a...         NaN   
2         7  Route 117 inbound trips all day on Saturdays a...         NaN   
3         7  Route 117 inbound trips all day on Saturdays a...         NaN   
4         7  Route 117 inbound trips all day on Saturdays a...         NaN   

  alert_lifecycle  ... active_period_start_date active_period_

**Merging all existing alert files**:

In [5]:
directory_now = '../../data/raw-data/Alerts_2024'
print(os.path.exists(directory_now))    # debug statements 

directory_archive = '../../data/raw-data/Archive_2020-2023'
print(os.path.exists(directory_archive))     # debug statements 

dataframes = [] # list initializaton for dataframes

# function: append files in directories, add the year column
def append_data(directory_path): 
    for foldername, _, filenames in os.walk(directory_path):
        for filename in filenames:
            if filename.endswith('.csv'): 
                file_path = os.path.join(foldername, filename)

                df = pd.read_csv(file_path) # load csv files 

                # Extract the year for yr column, add to dataset
                if '-' in filename:                                 # For 2024 files, due to formatting dif
                    year = int(filename.split('-')[0]) 
                elif '_' in filename:                               # For 2020-2023 files 
                    year = int(filename.split('_')[1]) 
                else:
                    raise ValueError(f"Unexpected filename format: {filename}")
               
                df['year'] = year

                # Append the DataFrames to the list
                dataframes.append(df)

# Load files from directories
append_data(directory_now)
append_data(directory_archive)

# Concatenate all DataFrames into a single DataFrame
merged_data = pd.concat(dataframes, ignore_index=True)

# Save the merged DataFrame to a new CSV file
output_file = '../../data/raw-data/merged_alerts.csv'
merged_data.to_csv(output_file, index=False)

print(f"Output saved as '{output_file}'.")



True
True


/var/folders/wx/8tzhq3ln59j92r3pl8v47rfw0000gn/T/ipykernel_41971/2374957634.py:16: DtypeWarning: Columns (16,25) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path) # load csv files
/var/folders/wx/8tzhq3ln59j92r3pl8v47rfw0000gn/T/ipykernel_41971/2374957634.py:16: DtypeWarning: Columns (16,25) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path) # load csv files
/var/folders/wx/8tzhq3ln59j92r3pl8v47rfw0000gn/T/ipykernel_41971/2374957634.py:16: DtypeWarning: Columns (13,16,25) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path) # load csv files
/var/folders/wx/8tzhq3ln59j92r3pl8v47rfw0000gn/T/ipykernel_41971/2374957634.py:16: DtypeWarning: Columns (16,25) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path) # load csv files
/var/folders/wx/8tzhq3ln59j92r3pl8v47rfw0000gn/T/ipykernel_41

Output saved as '../../data/raw-data/merged_alerts.csv'.


**After Merging**:

In [8]:
file_path = '../../data/raw-data/merged_alerts.csv' 

df = pd.read_csv(file_path)

shape = df.shape
print(shape) 

/var/folders/wx/8tzhq3ln59j92r3pl8v47rfw0000gn/T/ipykernel_75791/1535254680.py:3: DtypeWarning: Columns (16,25) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)


(2736828, 28)


## Current News Media 
This script references starter code from Homework 3 DSAN 5000.
* Authored by James Hickman. 
* Modified and extended by Mandy Sun.

In [ ]:
# Load API key from personal JSON file
with open('/Users/mandy.sun/Desktop/api_keys/.api-keys.json') as f:
    keys = json.load(f)
API_KEY = keys['serpapi']  # Load SerpApi key from personal JSON file; named it serpapi

# output file 
output_file = '../../data/raw-data/google_news_results.json'

# list of search queries
queries = ["MBTA", "Massachusetts Bay Transportation Authority"]
combined_results = []   # Store results 

# Iterate through each query and fetch results from Google News
for query in queries:
    print(f"Performing Google News search for: {query}")
    
    # Define search parameters for Google News
    params = {
        "engine": "google_news",      # use google news engine on serpapi 
        "q": query,                   # Search query
        "hl": "en",                   # Language: eng
        "gl": "us",                   # location: US 
        "num": 20,                    # results per page 
        "api_key": API_KEY            # Key
    }
    search = GoogleSearch(params)
    result_dict = search.get_dict()

    # append results to combined results list
    combined_results.extend(result_dict.get("news_results", []))

    # print the news articles to check 
    for result in result_dict.get("news_results", []):
        print(f"Title: {result.get('title')}")
        print(f"Link: {result.get('link')}")
        print(f"Source: {result.get('source')}")
        print("-" * 50)

# save to json 
with open(output_file, "w") as json_file:
    json.dump(combined_results, json_file, indent=4)

print(f"Combined news data from both queries saved to {output_file}")



Performing Google News search for: MBTA
Title: Slow zones to end across all MBTA lines for the first time in 22 years
Link: https://www.bostonherald.com/2024/12/15/slow-zones-to-end-across-all-mbta-lines-for-the-first-time-in-22-years/
Source: {'name': 'Boston Herald', 'icon': 'https://encrypted-tbn2.gstatic.com/faviconV2?url=https://www.bostonherald.com&client=NEWS_360&size=96&type=FAVICON&fallback_opts=TYPE,SIZE,URL', 'authors': ['Grace Zokovitch']}
--------------------------------------------------
Title: MBTA launches first phase of '﻿bus network redesign project'
Link: https://www.wcvb.com/article/mbta-proposes-new-bus-network-redesign-project/63195293
Source: {'name': 'WCVB Boston', 'icon': 'https://encrypted-tbn0.gstatic.com/faviconV2?url=https://www.wcvb.com&client=NEWS_360&size=96&type=FAVICON&fallback_opts=TYPE,SIZE,URL', 'authors': ['Danae Bucci']}
--------------------------------------------------
Title: Two more MBTA employees fired amid Cabot Yard investigation
Link: http

## Reliability 
For the sake of collection replication, we will show the first few lines of reliability dataset and its shape as reference. However, you can download the csv files yourself. 

In [20]:
example_data_reliability = "../../data/raw-data/MBTA_The_RIDE_Reliability.csv"

df2 = pd.read_csv(example_data_reliability)
shape = df2.shape
print(df2.head(5))
print(shape) 

                trip_date  ontime_trip_count  trip_count  ObjectId
0  2014/07/01 04:00:00+00               5479        5884         1
1  2014/07/02 04:00:00+00               5297        5612         2
2  2014/07/03 04:00:00+00               4959        5306         3
3  2014/07/04 04:00:00+00               1530        1558         4
4  2014/07/05 04:00:00+00               2420        2580         5
(3681, 4)


## Accuracy
For the sake of collection replication, we will show the first few lines of accurracy dataset and its shape as reference. However, you can download the csv files yourself. 

In [21]:
example_data_accuracy = "../../data/raw-data/rapid_transit_and_bus_prediction_accuracy_data.csv"

df3 = pd.read_csv(example_data_accuracy)
shape = df3.shape
print(df3.head(5))
print(shape) 

       weekly mode route_id        bin arrival_departure  num_predictions  \
0  2020-08-06  bus      NaN    0-3 min         departure           233967   
1  2020-08-06  bus      NaN    3-6 min         departure           227406   
2  2020-08-06  bus      NaN   6-12 min         departure           445848   
3  2020-08-06  bus      NaN  12-30 min         departure          1261843   
4  2020-08-13  bus      NaN    0-3 min         departure           240187   

   num_accurate_predictions  
0                    183615  
1                    178297  
2                    368421  
3                   1101363  
4                    189546  
(7964, 7)


## Financial Statement 

Arguably, the transformation from pdf to csv structured file lies in the intersection of collection and cleaning. However, we will be explaining the transformation into the cleaning page. 

**Before Transforming**:

Below is a linked example of the financial statement pdf file **before** converting to a structured format (.csv)

[View a Financial Statement PDF as an example](https://www.mass.gov/doc/fiscal-2023-massdot/download)


{{< include closing.qmd >}} 